# NorthStar - MongoDB Atlas design

## Aim
This notebook shows how MongoDB Atlas can support NorthStar's operational case tracking.

## Why this matters for the assignment
NorthStar has complaints, incidents, app events, and delivery exceptions that do not fit neatly into rigid relational tables.

A document model is useful because one order case can keep related records together in one place.

## Questions answered here
- Why is MongoDB suitable for this case?
- What should one order case document look like?
- How can the documents be uploaded into MongoDB Atlas?
- What indexes and queries support business investigation?

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT_DIR = Path('/content/assignment')
OUTPUT_DIR = ROOT_DIR / 'outputs_mongodb'
OUTPUT_DIR.mkdir(exist_ok=True)

DATE_COLUMNS = {
    'customers': ['signup_date'],
    'orders': ['order_created_at'],
    'deliveries': ['dispatch_time', 'delivery_completed_at'],
    'incidents': ['reported_at'],
    'complaints': ['created_at'],
    'app_events': ['event_timestamp'],
    'vehicles': ['commission_date'],
}

print('Repository folder:', ROOT_DIR)
print('MongoDB outputs folder:', OUTPUT_DIR)

In [ ]:
def load_table(name):
    df = pd.read_csv(ROOT_DIR / f'{name}.csv')
    for column_name in DATE_COLUMNS.get(name, []):
        df[column_name] = pd.to_datetime(df[column_name], errors='coerce')
    return df

customers = load_table('customers')
orders = load_table('orders')
deliveries = load_table('deliveries')
complaints = load_table('complaints')
incidents = load_table('incidents')
app_events = load_table('app_events')
drivers = load_table('drivers')
vehicles = load_table('vehicles')
hubs = load_table('hubs')

print('orders:', orders.shape)
print('deliveries:', deliveries.shape)
print('complaints:', complaints.shape)
print('incidents:', incidents.shape)
print('app_events:', app_events.shape)

In [ ]:
customer_lookup = customers.set_index('customer_id')
delivery_lookup = deliveries.set_index('order_id')
driver_lookup = drivers.set_index('driver_id')
vehicle_lookup = vehicles.set_index('vehicle_id')
hub_lookup = hubs.set_index('hub_id')

def safe_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value

def row_to_dict(row, columns):
    return {column: safe_value(row[column]) for column in columns if column in row.index}

def build_documents():
    documents = []

    for _, order_row in orders.iterrows():
        order_id = order_row['order_id']
        customer_id = order_row['customer_id']

        customer_row = customer_lookup.loc[customer_id] if customer_id in customer_lookup.index else None
        delivery_row = delivery_lookup.loc[order_id] if order_id in delivery_lookup.index else None

        if delivery_row is not None:
            delivery_id = delivery_row['delivery_id']
            driver_id = delivery_row['driver_id']
            vehicle_id = delivery_row['vehicle_id']
            hub_id = delivery_row['hub_id']
        else:
            delivery_id = None
            driver_id = None
            vehicle_id = None
            hub_id = None

        driver_row = driver_lookup.loc[driver_id] if driver_id in driver_lookup.index else None
        vehicle_row = vehicle_lookup.loc[vehicle_id] if vehicle_id in vehicle_lookup.index else None
        hub_row = hub_lookup.loc[hub_id] if hub_id in hub_lookup.index else None

        order_complaints = complaints[complaints['order_id'] == order_id].sort_values('created_at')
        order_incidents = incidents[incidents['delivery_id'] == delivery_id].sort_values('reported_at') if delivery_id else incidents.iloc[0:0]
        order_events = app_events[app_events['order_id'] == order_id].sort_values('event_timestamp')

        document = {
            '_id': order_id,
            'customer': row_to_dict(
                customer_row,
                ['customer_id', 'age', 'home_zone', 'customer_type', 'signup_date', 'loyalty_score', 'app_engagement_score', 'preferred_channel', 'account_status']
            ) if customer_row is not None else None,
            'order': row_to_dict(
                order_row,
                ['order_id', 'service_type', 'order_created_at', 'promised_window_hours', 'pickup_zone', 'dropoff_zone', 'priority_level', 'order_value', 'booking_channel', 'special_handling_flag']
            ),
            'delivery': row_to_dict(
                delivery_row,
                ['delivery_id', 'driver_id', 'vehicle_id', 'hub_id', 'dispatch_time', 'delivery_completed_at', 'delivery_status', 'route_distance_km', 'manual_route_override_count', 'proof_of_completion_missing', 'customer_rating_post_delivery', 'fuel_or_charge_cost']
            ) if delivery_row is not None else None,
            'driver': row_to_dict(
                driver_row,
                ['driver_id', 'base_zone', 'employment_type', 'years_experience', 'training_score', 'driver_rating', 'shift_preference', 'active_flag']
            ) if driver_row is not None else None,
            'vehicle': row_to_dict(
                vehicle_row,
                ['vehicle_id', 'vehicle_type', 'assigned_zone', 'commission_date', 'battery_health_pct', 'odometer_km', 'maintenance_status', 'telematics_version']
            ) if vehicle_row is not None else None,
            'hub': row_to_dict(
                hub_row,
                ['hub_id', 'hub_name', 'zone', 'hub_type', 'capacity_score']
            ) if hub_row is not None else None,
            'complaints': [
                row_to_dict(
                    complaint_row,
                    ['complaint_id', 'complaint_type', 'channel', 'severity', 'created_at', 'status', 'resolution_days', 'compensation_amount']
                )
                for _, complaint_row in order_complaints.iterrows()
            ],
            'incidents': [
                row_to_dict(
                    incident_row,
                    ['incident_id', 'incident_type', 'reported_at', 'severity', 'resolution_status', 'resolved_hours']
                )
                for _, incident_row in order_incidents.iterrows()
            ],
            'app_events': [
                row_to_dict(
                    event_row,
                    ['event_id', 'event_timestamp', 'event_type', 'session_id', 'device_type', 'zone_context', 'api_latency_ms', 'success_flag']
                )
                for _, event_row in order_events.iterrows()
            ],
        }

        document['risk_flags'] = {
            'late_or_failed': document['delivery'] is not None and document['delivery'].get('delivery_status') in ['Delayed', 'Failed'],
            'has_complaint': len(document['complaints']) > 0,
            'has_incident': len(document['incidents']) > 0,
            'manual_override_used': document['delivery'] is not None and (document['delivery'].get('manual_route_override_count') or 0) > 0,
        }

        documents.append(document)

    return documents

In [ ]:
documents = build_documents()

print('Total MongoDB-ready documents:', len(documents))
print('')
print('Example document preview:')
print(json.dumps(documents[0], indent=2, default=str))

jsonl_path = OUTPUT_DIR / 'order_cases.jsonl'
with jsonl_path.open('w', encoding='utf-8') as file_handle:
    for document in documents:
        file_handle.write(json.dumps(document, ensure_ascii=False) + '\n')

print('')
print('Saved JSONL file to:', jsonl_path)

## Why this document design is stronger for the assignment

This design is useful because it stores one full operational case per order.

That means the document keeps together:
- the customer
- the order details
- the delivery outcome
- the driver and vehicle context
- complaint history
- incident history
- app event history
- simple risk flags

This is stronger than separate flat tables when managers need to investigate one problem case from start to finish.

## Indexes to explain in the report

Useful indexes for this design are:
- `customer.customer_id`
- `order.service_type`
- `delivery.delivery_status`
- `hub.zone`
- `risk_flags.late_or_failed`

These indexes help managers search customer cases, filter by service type, review cases by zone, and quickly isolate delayed or failed work.

In [ ]:
# Run this cell only after you create a MongoDB Atlas cluster and copy the connection string.
# If needed, uncomment the next line in Colab.
# !pip -q install pymongo

from pymongo import MongoClient, ReplaceOne

MONGODB_URI = ''
DATABASE_NAME = 'northstar'
COLLECTION_NAME = 'order_cases'

if not MONGODB_URI:
    print('Paste your MongoDB Atlas connection string into MONGODB_URI before running this upload step.')
else:
    client = MongoClient(MONGODB_URI)
    collection = client[DATABASE_NAME][COLLECTION_NAME]

    operations = [ReplaceOne({'_id': document['_id']}, document, upsert=True) for document in documents]
    if operations:
        collection.bulk_write(operations)

    collection.create_index('customer.customer_id')
    collection.create_index('order.service_type')
    collection.create_index('delivery.delivery_status')
    collection.create_index('hub.zone')
    collection.create_index('risk_flags.late_or_failed')

    print(f'Uploaded {len(documents)} documents to {DATABASE_NAME}.{COLLECTION_NAME}')

## Example MongoDB queries for the report

These example queries make the MongoDB section stronger because they show how managers could actually use the database after it is built.

In [ ]:
# Query 1: delayed or failed cases with complaints
if 'collection' not in globals():
    print('Run the Atlas upload cell first after adding your connection string.')
else:
    delayed_cases = list(
        collection.find(
            {
                'risk_flags.late_or_failed': True,
                'risk_flags.has_complaint': True,
            },
            {
                '_id': 1,
                'customer.customer_id': 1,
                'order.service_type': 1,
                'hub.zone': 1,
                'delivery.delivery_status': 1,
                'complaints': {'$slice': 2},
            },
        ).limit(5)
    )
    print(json.dumps(delayed_cases, indent=2, default=str))

### Interpretation of query 1

This query is useful because it finds orders that combine poor delivery outcome and customer dissatisfaction.

That means a manager can review high-priority exception cases without joining many separate tables first.

In [ ]:
# Query 2: zones with the highest number of risky cases
if 'collection' not in globals():
    print('Run the Atlas upload cell first after adding your connection string.')
else:
    risky_zone_summary = list(
        collection.aggregate([
            {
                '$group': {
                    '_id': '$hub.zone',
                    'total_cases': {'$sum': 1},
                    'late_or_failed_cases': {
                        '$sum': {'$cond': ['$risk_flags.late_or_failed', 1, 0]}
                    },
                    'complaint_cases': {
                        '$sum': {'$cond': ['$risk_flags.has_complaint', 1, 0]}
                    },
                }
            },
            {
                '$sort': {
                    'late_or_failed_cases': -1,
                    'complaint_cases': -1,
                }
            }
        ])
    )
    print(json.dumps(risky_zone_summary, indent=2, default=str))

### Interpretation of query 2

This aggregation groups documents by zone and counts how many risky cases appear in each area.

It is useful because managers can move from single-case review to area-level monitoring without redesigning the document model.

In [ ]:
# Query 3: customers with the highest number of complaints
if 'collection' not in globals():
    print('Run the Atlas upload cell first after adding your connection string.')
else:
    repeat_complaint_customers = list(
        collection.aggregate([
            {
                '$project': {
                    'customer_id': '$customer.customer_id',
                    'customer_type': '$customer.customer_type',
                    'home_zone': '$customer.home_zone',
                    'complaint_count': {'$size': '$complaints'}
                }
            },
            {
                '$match': {
                    'complaint_count': {'$gt': 0}
                }
            },
            {
                '$group': {
                    '_id': {
                        'customer_id': '$customer_id',
                        'customer_type': '$customer_type',
                        'home_zone': '$home_zone'
                    },
                    'total_complaints': {'$sum': '$complaint_count'},
                    'orders_with_complaints': {'$sum': 1}
                }
            },
            {
                '$sort': {
                    'total_complaints': -1,
                    'orders_with_complaints': -1
                }
            },
            {
                '$limit': 10
            }
        ])
    )
    print(json.dumps(repeat_complaint_customers, indent=2, default=str))

### Interpretation of query 3

This query identifies customers with repeated complaint history across multiple orders.

That is useful for the assignment because it shows how MongoDB can support customer case tracking over time, not only single-order lookup.

## Final explanation for the report

This MongoDB design is useful because it stores the order, customer, delivery, complaints, incidents, and app events together as one case document.

That makes it easier to:
- review the full history of one order
- investigate service failures
- analyse complaint and incident patterns without many table joins
- support changing event structures over time
- query semi-structured data for operational case management

## Why this is a stronger assignment version

This notebook now does more than show a sample document. It also shows:
- a complete document-building process
- export to JSONL
- Atlas upload logic
- index creation
- example business queries
- plain-English interpretation of the MongoDB results

So this version fits the assignment better because it connects database design, implementation, and business use.